# Building Image Generation Applications (OpenAI)

There's more to LLMs than text generation. You can also generate images from text descriptions. Images as a modality are useful across MedTech, architecture, tourism, game development, marketing, and more. In this lesson we work with today's **GPT Image** models on the OpenAI platform.

## Learning goals

By the end of this lesson you'll be able to:

- Explain what image generation is and where it's useful.
- Understand the `gpt-image` model family and how it differs from the legacy DALL·E models.
- Build an image generation application and edit images.

## What is image generation?

Image generation models create images from a text prompt. Modern models such as `gpt-image` learn the relationship between text and images during training, then iteratively turn random noise into an image that matches your prompt.

Two well-known families of image models are:

- **`gpt-image` (OpenAI)** — the current generation used in this lesson. It supports text-to-image generation and image editing (inpainting with a mask).
- **Midjourney** — a popular third-party model with its own service and Discord-based workflow.

> The older OpenAI image models — **DALL·E 2** and **DALL·E 3** — are legacy. DALL·E 3 is no longer available for new deployments, and features like `create_variation` only existed in DALL·E 2. Use the `gpt-image` models for new applications.

> **Important:** `gpt-image` models return the generated image as **base64** (`b64_json`), not as a URL. Your code decodes the base64 string to bytes and saves it — there's no image URL to download.


## अपनी पहली छवि निर्माण एप्लिकेशन बनाना

तो एक छवि निर्माण एप्लिकेशन बनाने के लिए क्या चाहिए? आपको निम्नलिखित पुस्तकालयों की आवश्यकता है:

- **python-dotenv**, इस पुस्तकालय का उपयोग करने की अत्यधिक सिफारिश की जाती है ताकि आपके रहस्य एक *.env* फ़ाइल में कोड से दूर रखे जाएं।
- **openai**, यह पुस्तकालय है जिसका उपयोग आप OpenAI API के साथ इंटरैक्ट करने के लिए करेंगे।
- **pillow**, Python में छवियों के साथ काम करने के लिए।


1. एक *.env* फ़ाइल बनाएँ जिसमें निम्नलिखित सामग्री हो:

    ```text
    OPENAI_API_KEY='<add your OpenAI key here>'
    ```



1. ऊपर दी गई लाइब्रेरीज़ को *requirements.txt* नामक फ़ाइल में इस प्रकार संग्रहित करें:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. इसके बाद, वर्चुअल एनवायरनमेंट बनाएँ और लाइब्रेरीज़ इंस्टॉल करें:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> विंडोज़ के लिए, अपना वर्चुअल वातावरण बनाने और सक्रिय करने के लिए निम्नलिखित कमांड का उपयोग करें:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. *app.py* नामक फ़ाइल में निम्नलिखित कोड जोड़ें:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai

    # dotenv को इम्पोर्ट करें
    dotenv.load_dotenv()

    # OpenAI ऑब्जेक्ट बनाएँ (आपके .env से OPENAI_API_KEY पढ़ता है)
    client = openai.OpenAI()


    try:
        # इमेज जनरेशन API का उपयोग करके एक छवि बनाएं
        generation_response = client.images.generate(
            model="gpt-image-1",
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # यहां अपना प्रॉम्प्ट टेक्स्ट दर्ज करें
            size='1024x1024',
            n=1
        )
        # संग्रहीत छवि के लिए निर्देशिका सेट करें
        image_dir = os.path.join(os.curdir, 'images')

        # यदि निर्देशिका मौजूद नहीं है, तो इसे बनाएं
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # छवि पथ प्रारंभ करें (ध्यान दें कि फ़ाइल प्रकार png होना चाहिए)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # gpt-image मॉडल छवि को base64 (b64_json) के रूप में लौटाते हैं, URL के रूप में नहीं
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # डिफ़ॉल्ट इमेज व्यूअर में छवि दिखाएं
        image = Image.open(image_path)
        image.show()

    # अपवाद पकड़ें
    except openai.BadRequestError as err:
        print(err)

    ```

आइए इस कोड को समझते हैं:

- सबसे पहले, हम आवश्यक लाइब्रेरीज़ को इंपोर्ट करते हैं, जिनमें OpenAI लाइब्रेरी, dotenv लाइब्रेरी, base64 मॉड्यूल, और Pillow लाइब्रेरी शामिल हैं।

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai
    ```

- उसके बाद, हम क्लाइंट बनाते हैं, जो आपकी ``.env`` से API की को पढ़ता है।

    ```python
    # OpenAI ऑब्जेक्ट बनाएं
    client = openai.OpenAI()
    ```

- फिर, हम इमेज जनरेट करते हैं:

    ```python
    generation_response = client.images.generate(
        model="gpt-image-1",
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # यहाँ अपना प्रॉम्प्ट टेक्स्ट दर्ज करें
        size='1024x1024',
        n=1
    )
    ```

    `gpt-image` मॉडल इमेज को **base64** स्ट्रिंग के रूप में `data[0].b64_json` में लौटाते हैं। हम इसे बाइट्स में डिकोड करते हैं और एक फ़ाइल में लिखते हैं — डाउनलोड करने के लिए कोई URL नहीं होता।

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- अंत में, हम इमेज को खोलते हैं और इसे दिखाने के लिए स्टैंडर्ड इमेज व्यूअर का उपयोग करते हैं:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### इमेज जनरेट करने के बारे में और विवरण

आइए `images.generate` के पैरामीटर्स देखें:

- **model**, वह इमेज मॉडल है जिसे उपयोग किया जाता है (जैसे `gpt-image-1`)।
- **prompt**, वह टेक्स्ट प्रॉम्प्ट है जिससे इमेज जनरेट की जाती है। यहाँ यह "Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils" है।
- **size**, जनरेट की गई इमेज का आकार होता है (जैसे `1024x1024`, `1536x1024`, `1024x1536`, या `"auto"`)।
- **n**, जनरेट की जाने वाली इमेज की संख्या है। यहाँ हमने एक इमेज जनरेट की है।

> इमेज मॉडल `temperature` पैरामीटर नहीं लेते — यह एक टेक्स्ट-जेनरेशन नियंत्रण है। विविधता पाने के लिए, API को फिर से कॉल करें; विविधता कम करने के लिए, अपना प्रॉम्प्ट अधिक विशिष्ट बनाएं।

## इमेज जेनरेशन की अतिरिक्त क्षमताएं

आपने देखा कि कुछ पायथन लाइनों के साथ इमेज कैसे जनरेट की जाती है। `gpt-image` मॉडल मौजूदा इमेज को भी **संपादित** कर सकते हैं। एक इमेज, एक वैकल्पिक **मास्क** (जो बदलाव करने वाले क्षेत्र को चिह्नित करता है), और एक प्रॉम्प्ट देकर, आप इमेज के एक हिस्से को बदल सकते हैं — उदाहरण के लिए, हमारे खरगोश पर एक टोपी जोड़ना।

```python
response = client.images.edit(
  model="gpt-image-1",
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# संपादित को भी बेस64 के रूप में वापस किया जाता है
edited_image = base64.b64decode(response.data[0].b64_json)
```

बेस इमेज में केवल खरगोश होता है; अंतिम इमेज में खरगोश पर टोपी होती है।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
इस दस्तावेज़ का अनुवाद AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) का उपयोग करके किया गया है। जबकि हम सटीकता के लिए प्रयास करते हैं, कृपया ध्यान दें कि स्वचालित अनुवादों में त्रुटियाँ या अशुद्धियाँ हो सकती हैं। मूल दस्तावेज़ अपनी मूल भाषा में ही प्रामाणिक स्रोत माना जाना चाहिए। महत्वपूर्ण जानकारी के लिए, पेशेवर मानव अनुवाद की सिफारिश की जाती है। इस अनुवाद के उपयोग से उत्पन्न किसी भी गलतफहमी या गलत व्याख्या के लिए हम उत्तरदायी नहीं हैं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
